# DAT617 — NYC Flight Delays & Weather Analysis
## Full Pipeline: Download → Clean → Merge → Validate
**Team 7 | Chintan · Shivali · Tanisha | Prof. Vishal Lala**

| Stage | What happens |
|-------|-------------|
| §1 BTS Download | Pull 18 columns for JFK/LGA/EWR departures 2021–2026 directly from BTS |
| §2 Clean Flights | Parse dates, cap outliers, derive labels, build join key |
| §3 Weather | Load your pre-built weather CSV, build matching join key |
| §4 Merge | Left-join flights → weather on hour-level key |
| §5 Final | Column inventory, save train/test splits, summary |

> **Nothing is deleted without your approval.** Each section explains *why* every column exists.

In [ ]:
# ── Installs & Imports ──────────────────────────────────────────────
# requests + zipfile handle the BTS zip downloads
# Everything else is standard Colab

import pandas as pd
import numpy as np
import os, io, zipfile, requests, time, warnings
warnings.filterwarnings("ignore")

from google.colab import drive

print("✅ All libraries loaded")

In [ ]:
# ── Mount Drive & Set Paths ─────────────────────────────────────────
# WHY: Colab RAM resets every session. Google Drive is permanent.
#      All intermediate checkpoints and final CSVs save here
#      so a disconnect never loses your work.

drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Data Science Project"
RAW_DIR      = os.path.join(PROJECT_ROOT, "raw_data")
CLEAN_DIR    = os.path.join(PROJECT_ROOT, "cleaned_data")
VIZ_DIR      = os.path.join(PROJECT_ROOT, "visualizations")

for d in [RAW_DIR, CLEAN_DIR, VIZ_DIR]:
    os.makedirs(d, exist_ok=True)

# Your already-downloaded weather file (140,112 rows, all 3 airports)
WEATHER_CSV = os.path.join(RAW_DIR, "meteostat_weather_nyc_2021_2026_raw.csv")

NYC_AIRPORTS = ["JFK", "LGA", "EWR"]

print(f"✅ Project root : {PROJECT_ROOT}")
print(f"✅ Weather file : {'FOUND' if os.path.exists(WEATHER_CSV) else '❌ NOT FOUND — check filename'}")

---
## §1 — BTS Flight Data Download
**Why direct download instead of the web form?**
The web form requires selecting fields manually per month across 60 months — you
already experienced how tedious that is, and it's where `OP_CARRIER` got missed.
This cell pulls directly from BTS's pre-built zip archive, reads **only the 18
columns we need** (cuts memory ~80%), and filters to NYC departures **at read
time** so a 35M-row file never sits in RAM.

In [ ]:
# ── BTS Download Function ───────────────────────────────────────────
#
# COLUMN MAP — why each column is here:
#
#  YEAR/MONTH/DAY_OF_MONTH/DAY_OF_WEEK  → time controls for H3 (cascade),
#                                          seasonality, IS_WEEKEND flag
#  FL_DATE         → primary date column for all time-series and join key
#  Reporting_Airline → OP_CARRIER — THE column we missed before.
#                      Without it we cannot do H2 (legacy vs ULCC).
#                      In PREZIP files BTS calls it Reporting_Airline,
#                      not Op_Unique_Carrier (that name is for web form).
#  Origin / Dest   → filter to NYC; Dest kept for route-level analysis
#  DepTime         → source for DEP_HOUR (needed for weather join)
#  DepDelay        → raw delay — we NEVER overwrite originals
#  DepDelayMinutes → non-negative version (early = 0, not negative)
#  Cancelled / CancellationCode → outcome variable + reason (B=weather)
#  WeatherDelay / CarrierDelay / NASDelay → delay attribution breakdown
#  Distance        → control variable (longer flights = bigger buffers)
#  AirTime         → operational efficiency metric

BTS_COL_MAP = {
    "Year":              "YEAR",
    "Month":             "MONTH",
    "DayofMonth":        "DAY_OF_MONTH",
    "DayOfWeek":         "DAY_OF_WEEK",
    "FlightDate":        "FL_DATE",
    "Reporting_Airline": "OP_CARRIER",
    "Origin":            "ORIGIN",
    "Dest":              "DEST",
    "DepTime":           "DEP_TIME",
    "DepDelay":          "DEP_DELAY",
    "DepDelayMinutes":   "DEP_DELAY_NEW",
    "Cancelled":         "CANCELLED",
    "CancellationCode":  "CANCELLATION_CODE",
    "WeatherDelay":      "WEATHER_DELAY",
    "CarrierDelay":      "CARRIER_DELAY",
    "NASDelay":          "NAS_DELAY",
    "Distance":          "DISTANCE",
    "AirTime":           "AIR_TIME",
}
BTS_COLS_NEEDED = set(BTS_COL_MAP.keys())

def download_bts_month(year, month, retries=3):
    url = (
        f"https://transtats.bts.gov/PREZIP/"
        f"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
    )
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=180)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
                with z.open(csv_name) as f:
                    df = pd.read_csv(
                        f,
                        usecols=lambda c: c.strip() in BTS_COLS_NEEDED,
                        dtype={"CancellationCode": str},
                        low_memory=False
                    )
            df.columns = df.columns.str.strip()
            df.rename(columns=BTS_COL_MAP, inplace=True)
            # DEPARTURES ONLY — arrivals excluded (study is about departures)
            df = df[df["ORIGIN"].isin(NYC_AIRPORTS)].copy()
            return df, None
        except Exception as e:
            if attempt < retries:
                time.sleep(5)
            else:
                return None, str(e)

print("✅ Download function ready")
print(f"   Pulling {len(BTS_COL_MAP)} columns per month, NYC departures only")

In [ ]:
# ── Download 2021–2025 (60 months) ─────────────────────────────────
# WHY YEARLY CHECKPOINTS:
#   If Colab disconnects mid-download you lose everything.
#   We save a CSV per year so you can resume from the last
#   complete year instead of starting over.

all_frames_hist = []
failed = []

for year in range(2021, 2026):
    year_frames = []
    print(f"\n━━━ {year} ━━━")

    for month in range(1, 13):
        print(f"  {year}-{month:02d}...", end=" ", flush=True)
        df, err = download_bts_month(year, month)

        if df is not None:
            year_frames.append(df)
            print(f"✓ {len(df):,} rows")
        else:
            failed.append((year, month, err))
            print(f"✗ FAILED: {str(err)[:60]}")
        time.sleep(0.5)

    if year_frames:
        yr_df = pd.concat(year_frames, ignore_index=True)
        ckpt  = os.path.join(RAW_DIR, f"bts_nyc_{year}.csv")
        yr_df.to_csv(ckpt, index=False)
        all_frames_hist.append(yr_df)
        print(f"  💾 Checkpoint saved: bts_nyc_{year}.csv  ({len(yr_df):,} rows)")

if failed:
    print(f"\n⚠️  {len(failed)} month(s) failed: {failed}")
else:
    print(f"\n✅ All 60 months downloaded successfully")

In [ ]:
# ── Download 2026 (available months only) ───────────────────────────
# WHY 2026:
#   Train on 2021-2025 → predict 2026 → compare prediction to actual.
#   That is a true out-of-sample validation, much stronger than a
#   random 80/20 split on the same years.

all_frames_2026 = []
print("━━━ 2026 (trying Jan–May) ━━━")

for month in range(1, 6):
    print(f"  2026-{month:02d}...", end=" ", flush=True)
    df, err = download_bts_month(2026, month)
    if df is not None:
        all_frames_2026.append(df)
        print(f"✓ {len(df):,} rows")
    else:
        print(f"✗ Not available yet")
    time.sleep(0.5)

if all_frames_2026:
    df_2026 = pd.concat(all_frames_2026, ignore_index=True)
    ckpt_26 = os.path.join(RAW_DIR, "bts_nyc_2026.csv")
    df_2026.to_csv(ckpt_26, index=False)
    print(f"\n💾 Saved: bts_nyc_2026.csv  ({len(df_2026):,} rows)")
else:
    df_2026 = pd.DataFrame()
    print("\n⚠️  No 2026 data available — continuing with 2021-2025")

In [ ]:
# ── Combine All Downloaded Months ───────────────────────────────────
# WHY WE KEEP 2026 IN THE SAME DATAFRAME FOR NOW:
#   Cleaning steps (date parsing, outlier capping, feature engineering)
#   need to run on the full dataset consistently. We will split
#   train vs test AFTER cleaning, before modeling.

print("Combining all months...")

all_frames = all_frames_hist + (all_frames_2026 if all_frames_2026 else [])
flights_raw = pd.concat(all_frames, ignore_index=True)

print(f"\n✅ Combined: {len(flights_raw):,} rows × {flights_raw.shape[1]} columns")
print(f"\nAll columns present:")
for i, col in enumerate(flights_raw.columns, 1):
    print(f"  {i:02d}. {col}")

In [ ]:
# ════════════════════════════════════════════════════════
# SANITY CHECK 1 — Layout, Coverage, Column Presence
# ════════════════════════════════════════════════════════

print("=" * 58)
print("SANITY CHECK 1: Layout and Coverage")
print("=" * 58)

# 1a. All expected columns present?
missing_cols = [c for c in BTS_COL_MAP.values() if c not in flights_raw.columns]
if missing_cols:
    print(f"❌ MISSING COLUMNS: {missing_cols}")
else:
    print(f"✅ All {len(BTS_COL_MAP)} expected columns present")

# 1b. OP_CARRIER specifically (the critical missing column from before)
if "OP_CARRIER" in flights_raw.columns:
    unique_carriers = flights_raw["OP_CARRIER"].nunique()
    print(f"✅ OP_CARRIER present — {unique_carriers} unique carriers")
    print(f"   Sample: {sorted(flights_raw['OP_CARRIER'].dropna().unique())[:10]}")
else:
    print("❌ OP_CARRIER MISSING — check BTS_COL_MAP")

# 1c. Airport and year coverage
print(f"\nDepartures by airport:")
print(flights_raw["ORIGIN"].value_counts().reindex(NYC_AIRPORTS))
print(f"\nFlights by year:")
print(flights_raw["YEAR"].value_counts().sort_index())

# 1d. Quick data preview
print(f"\nFirst 3 rows (transposed for readability):")
print(flights_raw.head(3).T)

---
## §2 — Clean & Engineer Features
Each step below is explained inline. **No columns are deleted** —
questionable ones are flagged for your review in §5.

In [ ]:
# ── Parse Dates & Extract Departure Hour ────────────────────────────
# WHY DEP_HOUR:
#   Weather data is hourly. To attach the correct weather reading
#   to each flight we need the departure hour, not just the date.
#   E.g. a 14:35 departure gets the 14:00 weather reading.
#   DEP_TIME is stored as HHMM float (1435.0) → integer divide by 100.
#
# WHY WE KEEP DAY_OF_MONTH AND FL_DATE:
#   FL_DATE is the primary join key component. DAY_OF_MONTH is needed
#   to reconstruct dates if FL_DATE ever gets corrupted, and for
#   day-of-month trend analysis.

flights = flights_raw.copy()

# Parse date
flights["FL_DATE"] = pd.to_datetime(flights["FL_DATE"])

# Extract departure hour (cancelled flights have null DEP_TIME → null DEP_HOUR)
flights["DEP_HOUR"] = (flights["DEP_TIME"] // 100).astype("Int64")

# Extract month from date as backup (keeps MONTH col too — both stay)
flights["FL_MONTH"] = flights["FL_DATE"].dt.month

print(f"✅ FL_DATE parsed: {flights['FL_DATE'].min().date()} → {flights['FL_DATE'].max().date()}")
print(f"✅ DEP_HOUR sample (active flights): {flights['DEP_HOUR'].dropna().head(5).tolist()}")
print(f"✅ Null DEP_HOUR count: {flights['DEP_HOUR'].isnull().sum():,}  (expected ≈ cancelled flights)")
print(f"   Cancelled count:     {int(flights['CANCELLED'].sum()):,}")

In [ ]:
# ── Outlier Capping → DEP_DELAY_CLEAN ───────────────────────────────
# WHY WE CAP INSTEAD OF DROP:
#   Flights delayed 1,000+ minutes exist — they're real events
#   (diversions, ground stops). Dropping them would bias the
#   cancellation analysis. Capping preserves the row while
#   preventing one extreme value from dragging every mean and
#   regression coefficient. The raw DEP_DELAY column is untouched.
#
# WHY 99th PERCENTILE:
#   Standard practice for right-skewed distributions. Keeps 99%
#   of real variation intact while removing statistical noise
#   from the extreme tail.

active_mask = flights["CANCELLED"] == 0
p99 = flights.loc[active_mask, "DEP_DELAY"].quantile(0.99)
outlier_n = int((flights.loc[active_mask, "DEP_DELAY"] > p99).sum())

# Create capped column — original DEP_DELAY unchanged
flights["DEP_DELAY_CLEAN"] = flights["DEP_DELAY"].clip(upper=p99)

print(f"✅ 99th percentile cap : {p99:.0f} minutes")
print(f"✅ Outliers capped     : {outlier_n:,} rows (values replaced, rows KEPT)")
print(f"✅ Raw max             : {flights['DEP_DELAY'].max():.0f} min → Capped max: {flights['DEP_DELAY_CLEAN'].max():.0f} min")
print(f"✅ Total rows unchanged: {len(flights):,}")

In [ ]:
# ── Outcome Labels, Season, Weekend, Time-of-Day ────────────────────
# WHY EACH COLUMN:
#
# OUTCOME      → Four-way categorical used in every chart and as the
#                classification target. Replaces three separate binary
#                flags in a single readable column.
#
# SEASON       → Winter storms vs summer thunderstorms behave very
#                differently. Season is a confounder for weather effects
#                (H1). Must be controlled for in regression.
#
# IS_WEEKEND   → Business travelers dominate weekdays; leisure on
#                weekends. Load profile affects ground crew scheduling
#                and gate availability — control variable for H3.
#
# TIME_OF_DAY  → Central to H3 (cascade hypothesis). Even without
#                weather worsening, delays compound through the day.
#                Evening flights are delayed more than morning flights
#                under identical conditions — this column lets us test it.
#
# DELAYED_15 / DELAYED_60 / ON_TIME → FAA standard thresholds.
#                Used for on-time rate KPIs and H2 comparison charts.

# ── OUTCOME ──────────────────────────────────────────────────────────
conditions = [
    (flights["CANCELLED"] == 1),
    (flights["DEP_DELAY_CLEAN"] >= 60) & (flights["CANCELLED"] == 0),
    (flights["DEP_DELAY_CLEAN"] >= 15) & (flights["CANCELLED"] == 0),
    (flights["DEP_DELAY_CLEAN"] <  15) & (flights["CANCELLED"] == 0),
]
choices = ["Cancelled", "Severe Delay (60+ min)", "Delayed (15-59 min)", "On Time"]
flights["OUTCOME"] = np.select(conditions, choices, default="Unknown")

# ── SEASON ────────────────────────────────────────────────────────────
season_map = {
    12:"Winter",1:"Winter",2:"Winter",
    3:"Spring",4:"Spring",5:"Spring",
    6:"Summer",7:"Summer",8:"Summer",
    9:"Fall",10:"Fall",11:"Fall"
}
flights["SEASON"] = flights["MONTH"].map(season_map)

# ── IS_WEEKEND ────────────────────────────────────────────────────────
# BTS DAY_OF_WEEK: 1=Mon, 2=Tue, ..., 6=Sat, 7=Sun
flights["IS_WEEKEND"] = (flights["DAY_OF_WEEK"] >= 6).astype(int)

# ── TIME_OF_DAY ───────────────────────────────────────────────────────
def time_bucket(h):
    if pd.isna(h): return "Unknown"
    h = int(h)
    if h < 6:  return "Red-Eye (0-5)"
    if h < 9:  return "Early Morning (6-8)"
    if h < 12: return "Morning (9-11)"
    if h < 15: return "Early Afternoon (12-14)"
    if h < 18: return "Late Afternoon (15-17)"
    if h < 21: return "Evening (18-20)"
    return "Night (21-23)"

flights["TIME_OF_DAY"] = flights["DEP_HOUR"].apply(time_bucket)

# ── BINARY FLAGS ──────────────────────────────────────────────────────
flights["DELAYED_15"] = ((flights["DEP_DELAY"] >= 15) & (flights["CANCELLED"]==0)).astype(int)
flights["DELAYED_60"] = ((flights["DEP_DELAY"] >= 60) & (flights["CANCELLED"]==0)).astype(int)
flights["ON_TIME"]    = ((flights["DEP_DELAY"] <  15) & (flights["CANCELLED"]==0)).astype(int)

print("✅ Derived columns created:")
print(f"\nOUTCOME breakdown:")
print(flights["OUTCOME"].value_counts())
print(f"\nSEASON breakdown:")
print(flights["SEASON"].value_counts())
print(f"\nWeekend flights: {flights['IS_WEEKEND'].mean()*100:.1f}%")

In [ ]:
# ── Airline Tier Classification ─────────────────────────────────────
# WHY THIS COLUMN:
#   H2 tests whether ULCC carriers show greater weather sensitivity
#   than Legacy carriers. Raw carrier codes (AA, NK, B6...) can't
#   answer that — we need a tier label to group them.
#
# WHY WE FLAG UNMAPPED CARRIERS INSTEAD OF DROPPING:
#   Some carriers appear at NYC airports only for charter or seasonal
#   routes and aren't in standard classification lists. We label them
#   "Other" and print them so you can decide — don't silently lose rows.
#
# SOURCE: Matches the proposal Section 4 carrier table.

CARRIER_TIER = {
    # ── Domestic Legacy ──────────────────────────────────
    "AA":("Domestic","Legacy"),  "DL":("Domestic","Legacy"),
    "UA":("Domestic","Legacy"),
    # ── Domestic LCC ─────────────────────────────────────
    "B6":("Domestic","LCC"),    "WN":("Domestic","LCC"),
    "AS":("Domestic","LCC"),    "SY":("Domestic","LCC"),
    "VX":("Domestic","LCC"),
    # ── Domestic ULCC ────────────────────────────────────
    "NK":("Domestic","ULCC"),   "F9":("Domestic","ULCC"),
    "G4":("Domestic","ULCC"),
    # ── Domestic Regional ────────────────────────────────
    "OO":("Domestic","Regional"), "YX":("Domestic","Regional"),
    "MQ":("Domestic","Regional"), "9E":("Domestic","Regional"),
    "OH":("Domestic","Regional"), "QX":("Domestic","Regional"),
    "CP":("Domestic","Regional"), "C5":("Domestic","Regional"),
    "AX":("Domestic","Regional"), "PT":("Domestic","Regional"),
    "ZW":("Domestic","Regional"), "EM":("Domestic","Regional"),
    "G7":("Domestic","Regional"), "3M":("Domestic","Regional"),
    # ── International Legacy ─────────────────────────────
    "BA":("International","Legacy"), "EK":("International","Legacy"),
    "LH":("International","Legacy"), "SQ":("International","Legacy"),
    "QR":("International","Legacy"), "AF":("International","Legacy"),
    "KL":("International","Legacy"), "IB":("International","Legacy"),
    "AZ":("International","Legacy"), "TK":("International","Legacy"),
    "ET":("International","Legacy"), "MS":("International","Legacy"),
    "RJ":("International","Legacy"), "GF":("International","Legacy"),
    "AI":("International","Legacy"), "SA":("International","Legacy"),
    "KE":("International","Legacy"), "NH":("International","Legacy"),
    "CX":("International","Legacy"), "OZ":("International","Legacy"),
    "VS":("International","Legacy"), "AC":("International","Legacy"),
    "JL":("International","Legacy"), "MU":("International","Legacy"),
    "CA":("International","Legacy"), "BR":("International","Legacy"),
    "LA":("International","Legacy"), "AV":("International","Legacy"),
    "AM":("International","Legacy"), "PR":("International","Legacy"),
    "LX":("International","Legacy"), "OS":("International","Legacy"),
    "SK":("International","Legacy"), "AY":("International","Legacy"),
    "TG":("International","Legacy"), "GA":("International","Legacy"),
    "MH":("International","Legacy"), "SV":("International","Legacy"),
    "CI":("International","Legacy"), "UL":("International","Legacy"),
    # ── International LCC ────────────────────────────────
    "WS":("International","LCC"),  "FI":("International","LCC"),
    "EI":("International","LCC"),  "TP":("International","LCC"),
    "CM":("International","LCC"),  "TO":("International","LCC"),
    "XL":("International","LCC"),
    # ── International ULCC ───────────────────────────────
    "BF":("International","ULCC"), "XP":("International","ULCC"),
}

flights["FLIGHT_TYPE"] = flights["OP_CARRIER"].map(
    lambda x: CARRIER_TIER.get(x, ("Other","Other"))[0]
)
flights["AIRLINE_TIER"] = flights["OP_CARRIER"].map(
    lambda x: CARRIER_TIER.get(x, ("Other","Other"))[1]
)

unmapped = sorted(flights[flights["FLIGHT_TYPE"]=="Other"]["OP_CARRIER"].dropna().unique())

print("✅ FLIGHT_TYPE and AIRLINE_TIER created")
print(f"\nFlight type breakdown:")
print(flights["FLIGHT_TYPE"].value_counts())
print(f"\nAirline tier breakdown:")
print(flights["AIRLINE_TIER"].value_counts())

if unmapped:
    print(f"\n⚠️  {len(unmapped)} carrier(s) not in map → labelled 'Other':")
    print(f"   {unmapped}")
    print("   → Add them to CARRIER_TIER above if needed")
else:
    print("\n✅ All carriers mapped — no 'Other' entries")

In [ ]:
# ── Build Weather Join Key ───────────────────────────────────────────
# WHY HOUR-LEVEL PRECISION:
#   Daily average weather would miss morning fog that clears by noon,
#   or a storm arriving at 3pm. Hour-level join captures conditions
#   at the moment of departure — what actually causes the delay.
#
# FORMAT: "JFK_2021-03-15_14"
#   = airport + date + departure hour (zero-padded)
#
# WHY WE BUILD IT FOR CANCELLED FLIGHTS TOO:
#   Cancellations happen because of weather. We need to join
#   the weather that caused the cancellation to each cancelled row.
#   Flights with null DEP_HOUR (rare edge cases) will get a null
#   key and simply won't match — no rows are dropped.

flights["WEATHER_KEY"] = (
    flights["ORIGIN"] + "_" +
    flights["FL_DATE"].dt.strftime("%Y-%m-%d") + "_" +
    flights["DEP_HOUR"].astype(str).str.zfill(2)
)

null_keys = flights["WEATHER_KEY"].str.contains("<NA>", na=True).sum()
print(f"✅ WEATHER_KEY created")
print(f"   Sample: {flights['WEATHER_KEY'].dropna().head(3).tolist()}")
print(f"   Null keys (no DEP_TIME): {null_keys:,}")
print(f"   Valid keys             : {flights['WEATHER_KEY'].notna().sum():,}")

In [ ]:
# ════════════════════════════════════════════════════════
# SANITY CHECK 2 — Full Flights Dataframe Review
# Run before merge to confirm all columns and stats
# look correct. Nothing is dropped here.
# ════════════════════════════════════════════════════════

print("=" * 60)
print("SANITY CHECK 2: Flights Dataframe Review")
print("=" * 60)

print(f"\n📐 Shape: {flights.shape[0]:,} rows × {flights.shape[1]} columns")

print(f"\n📋 Column inventory:")
print(f"{'#':<4} {'Column':<25} {'Dtype':<14} {'Nulls':>8}  {'Null%':>6}  Note")
print("-" * 75)
for i, col in enumerate(flights.columns, 1):
    nulls = flights[col].isnull().sum()
    pct   = nulls / len(flights) * 100
    dtype = str(flights[col].dtype)
    flag  = "⚠️  HIGH" if pct > 20 else ("(cancelled)" if "DELAY" in col and pct > 1 else "")
    print(f"{i:<4} {col:<25} {dtype:<14} {nulls:>8,}  {pct:>5.1f}%  {flag}")

print(f"\n📊 Key metrics:")
active = flights[flights["CANCELLED"] == 0]
print(f"  Total flights       : {len(flights):,}")
print(f"  Active (not cancel) : {len(active):,}")
print(f"  Cancelled           : {int(flights['CANCELLED'].sum()):,}  ({flights['CANCELLED'].mean()*100:.1f}%)")
print(f"  On-time rate        : {active['ON_TIME'].mean()*100:.1f}%")
print(f"  Mean delay (active) : {active['DEP_DELAY'].mean():.1f} min")

print(f"\n📅 Date range: {flights['FL_DATE'].min().date()} → {flights['FL_DATE'].max().date()}")
print(f"\n✈️  Airport counts (departures):")
print(flights["ORIGIN"].value_counts().reindex(NYC_AIRPORTS))
print(f"\n🏢 Carrier tier counts:")
print(flights["AIRLINE_TIER"].value_counts())

---
## §3 — Weather Data
Your `meteostat_weather_nyc_2021_2026_raw.csv` is already complete:
- 140,112 rows (JFK + LGA + EWR, hourly, Jan 2021 → Apr 2026)
- Visibility patched from NOAA ASOS (365 nulls, 0.26% — acceptable)
- All other columns zero nulls

We just load it and build a matching join key.

In [ ]:
# ── Load Weather CSV ────────────────────────────────────────────────

print("Loading weather data...")
weather = pd.read_csv(WEATHER_CSV)

print(f"✅ Loaded: {len(weather):,} rows × {weather.shape[1]} columns")
print(f"\nColumns : {weather.columns.tolist()}")
print(f"Airports: {weather['AIRPORT'].unique().tolist()}")
print(f"Date range: {weather['time'].min()} → {weather['time'].max()}")
print(f"\nNull counts per column:")
print(weather.isnull().sum())

In [ ]:
# ── Build Weather Join Key ───────────────────────────────────────────
# WHY UTC:
#   The weather timestamps are in UTC (that's how NOAA/Open-Meteo
#   delivers them). BTS flight times are local (Eastern).
#   To align correctly we need to convert the weather key to
#   Eastern Time so "JFK_2021-03-15_14" means 2pm Eastern,
#   not 2pm UTC (which would be 10am Eastern — 4 hours off).

weather["time_parsed"] = pd.to_datetime(weather["time"], utc=True)

# Convert UTC → US/Eastern (handles DST automatically)
try:
    weather["time_eastern"] = weather["time_parsed"].dt.tz_convert("US/Eastern")
except Exception:
    # Fallback: subtract 5 hours (ignores DST — slightly less accurate)
    weather["time_eastern"] = weather["time_parsed"] - pd.Timedelta(hours=5)

weather["WEATHER_KEY"] = (
    weather["AIRPORT"] + "_" +
    weather["time_eastern"].dt.strftime("%Y-%m-%d") + "_" +
    weather["time_eastern"].dt.hour.astype(str).str.zfill(2)
)

print(f"✅ WEATHER_KEY built on weather side")
print(f"   UTC sample    : {weather['time'].iloc[0]}")
print(f"   Eastern sample: {weather['time_eastern'].iloc[0]}")
print(f"   Key sample    : {weather['WEATHER_KEY'].iloc[0]}")
print(f"   Total keys    : {len(weather):,}")

In [ ]:
# ════════════════════════════════════════════════════════
# SANITY CHECK 3 — Key Overlap Before Merge
# If coverage is below 90% the merge will produce mostly
# nulls. Catch it here before wasting time on the merge.
# ════════════════════════════════════════════════════════

print("=" * 58)
print("SANITY CHECK 3: Key Overlap Pre-Merge")
print("=" * 58)

flight_keys  = set(flights["WEATHER_KEY"].dropna())
weather_keys = set(weather["WEATHER_KEY"])
overlap      = flight_keys & weather_keys
coverage     = len(overlap) / len(flight_keys) * 100

print(f"  Flight keys (non-null) : {len(flight_keys):,}")
print(f"  Weather keys           : {len(weather_keys):,}")
print(f"  Matching keys          : {len(overlap):,}")
print(f"  Coverage               : {coverage:.1f}%")

if coverage >= 90:
    print(f"\n✅ Coverage ≥ 90% — ready to merge")
elif coverage >= 70:
    print(f"\n⚠️  Coverage {coverage:.1f}% — marginal, check timezone alignment")
    print(f"   Sample flight key : {list(flight_keys)[:2]}")
    print(f"   Sample weather key: {list(weather_keys)[:2]}")
else:
    print(f"\n❌ Coverage {coverage:.1f}% — keys are misaligned, do not merge yet")
    print(f"   Sample flight key : {list(flight_keys)[:3]}")
    print(f"   Sample weather key: {list(weather_keys)[:3]}")
    print("   → Check timezone conversion in the weather key cell above")

---
## §4 — Merge Flights + Weather

In [ ]:
# ── Merge Flights → Weather (Left Join) ─────────────────────────────
# WHY LEFT JOIN (not inner):
#   Inner join silently drops flights with no weather match
#   (e.g. cancelled flights with null DEP_HOUR, rare edge cases).
#   Left join keeps every flight. Those with no weather match
#   get NaN for weather columns — we can count and handle them
#   explicitly rather than losing rows invisibly.
#
# WHY WE DON'T DROP WEATHER_KEY YET:
#   We verify the merge result first. You approve dropping it in §5.

WEATHER_MERGE_COLS = [
    "WEATHER_KEY", "TEMP_C", "PRECIP_MM", "WIND_SPEED_KMH",
    "WIND_GUST_KMH", "VISIBILITY_KM", "SNOW_MM", "PRESSURE_HPA"
]

print("Merging...")
rows_before = len(flights)

df_merged = flights.merge(
    weather[WEATHER_MERGE_COLS],
    on="WEATHER_KEY",
    how="left"
)

rows_after   = len(df_merged)
missing_wx   = int(df_merged["TEMP_C"].isnull().sum())
missing_pct  = missing_wx / rows_after * 100

print(f"\n✅ Merge complete")
print(f"   Rows before : {rows_before:,}")
print(f"   Rows after  : {rows_after:,}  {'✅ unchanged' if rows_before==rows_after else '❌ CHANGED'}")
print(f"   Missing wx  : {missing_wx:,}  ({missing_pct:.1f}%)")

In [ ]:
# ── Severity Score & Adverse Weather Flag ───────────────────────────
# WHY SEVERITY_SCORE:
#   Individual weather variables (precip, wind, snow, visibility) are
#   correlated with each other. A composite score gives a single
#   continuous predictor for regression and makes charts readable.
#   Scale is 0–10 so coefficients are interpretable.
#
# SCORING (vectorized — fast on 2M+ rows):
#   Precipitation : 0–4 pts  (peaks at 20mm/hr — heavy rain/snow)
#   Wind Speed    : 0–3 pts  (peaks at 80 km/h — storm-force)
#   Snowfall      : 0–3 pts  (peaks at 15mm/hr — heavy snow)
#   Visibility    : +0.5 pts if <5km, +2 pts if <1km (severe fog)
#
# WHY ADVERSE_WEATHER BINARY FLAG:
#   For charts and H2 comparisons we need a clean yes/no split.
#   Threshold: any single variable crosses an operationally
#   meaningful level (same logic dispatchers use).

# ── Vectorized severity score ─────────────────────────────────────────
s = (
    df_merged["PRECIP_MM"].fillna(0).clip(upper=20)   / 5.0   +   # 0–4
    df_merged["WIND_SPEED_KMH"].fillna(0).clip(upper=80) / 26.67 + # 0–3
    df_merged["SNOW_MM"].fillna(0).clip(upper=15)     / 5.0         # 0–3
)
vis = df_merged["VISIBILITY_KM"].fillna(99)
s += (vis < 1.0).astype(float) * 2.0
s += ((vis >= 1.0) & (vis < 5.0)).astype(float) * 0.5
df_merged["SEVERITY_SCORE"] = s.clip(upper=10.0).round(2)

# ── Adverse weather binary flag ───────────────────────────────────────
df_merged["ADVERSE_WEATHER"] = (
    (df_merged["PRECIP_MM"].fillna(0)      >  2.5) |
    (df_merged["WIND_SPEED_KMH"].fillna(0) > 30.0) |
    (df_merged["SNOW_MM"].fillna(0)        >  0.5) |
    (df_merged["VISIBILITY_KM"].fillna(99) <  5.0)
).astype(int)

print(f"✅ SEVERITY_SCORE  : {df_merged['SEVERITY_SCORE'].min():.1f} – {df_merged['SEVERITY_SCORE'].max():.1f}  (mean {df_merged['SEVERITY_SCORE'].mean():.2f})")
print(f"✅ ADVERSE_WEATHER : {df_merged['ADVERSE_WEATHER'].mean()*100:.1f}% of all flight-hours flagged as adverse")

In [ ]:
# ════════════════════════════════════════════════════════
# SANITY CHECK 4 — Post-Merge Full Review
# ════════════════════════════════════════════════════════

print("=" * 60)
print("SANITY CHECK 4: Post-Merge Review")
print("=" * 60)

# ── Row count must not change ─────────────────────────────────────────
assert len(df_merged) == len(flights), "❌ Row count changed after merge!"
print(f"✅ Row count unchanged: {len(df_merged):,}")

# ── Weather coverage ──────────────────────────────────────────────────
missing_pct = df_merged["TEMP_C"].isnull().mean() * 100
status = "✅" if missing_pct < 5 else "⚠️"
print(f"{status} Missing weather: {missing_pct:.1f}%  ({'acceptable' if missing_pct < 5 else 'HIGH — check keys'})")

# ── Severity score range ──────────────────────────────────────────────
sev_ok = df_merged["SEVERITY_SCORE"].between(0, 10).all()
print(f"{'✅' if sev_ok else '❌'} Severity score in 0–10: {sev_ok}")

# ── Per-airport stats ─────────────────────────────────────────────────
print(f"\n📊 Per-airport summary:")
print(f"{'Airport':<8} {'Flights':>8} {'Cancel%':>8} {'OnTime%':>8} {'MeanDelay':>10} {'AdvWx%':>8}")
print("-" * 55)
for ap in NYC_AIRPORTS:
    sub = df_merged[df_merged["ORIGIN"] == ap]
    act = sub[sub["CANCELLED"] == 0]
    print(f"{ap:<8} {len(sub):>8,} {sub['CANCELLED'].mean()*100:>7.1f}% "
          f"{act['ON_TIME'].mean()*100:>7.1f}% "
          f"{act['DEP_DELAY'].mean():>9.1f}m "
          f"{sub['ADVERSE_WEATHER'].mean()*100:>7.1f}%")

# ── Tier-level stats ──────────────────────────────────────────────────
print(f"\n🏢 Per-tier summary (key for H2):")
tier_stats = df_merged.groupby("AIRLINE_TIER").agg(
    flights=("FL_DATE","count"),
    cancel_pct=("CANCELLED","mean"),
    mean_delay=("DEP_DELAY_CLEAN","mean"),
    adverse_pct=("ADVERSE_WEATHER","mean")
).round(3)
print(tier_stats)

# ── Year split check ──────────────────────────────────────────────────
print(f"\n📅 Year split (for train/test):")
print(df_merged["YEAR"].value_counts().sort_index())

---
## §5 — Final Column Inventory & Save
Review every column below. **Nothing is deleted here.**
Columns marked 🗂 ARCHIVE are candidates for removal — approve
in the next cell before anything is dropped.

In [ ]:
# ── Column Inventory — Review Before Any Deletions ──────────────────

COL_GUIDE = {
    "FL_DATE":           ("✅ KEEP",    "Primary date for all time series & join key"),
    "YEAR":              ("✅ KEEP",    "Train/test split boundary"),
    "MONTH":             ("✅ KEEP",    "Seasonal analysis and regression control"),
    "DAY_OF_MONTH":      ("✅ KEEP",    "Date reconstruction backup + day-of-month trends"),
    "DAY_OF_WEEK":       ("✅ KEEP",    "Source for IS_WEEKEND; weekly pattern analysis"),
    "FL_MONTH":          ("🗂 ARCHIVE", "Duplicate of MONTH — safe to drop after review"),
    "OP_CARRIER":        ("✅ KEEP",    "Carrier IATA code — essential for H2"),
    "ORIGIN":            ("✅ KEEP",    "Departure airport — essential for H4"),
    "DEST":              ("✅ KEEP",    "Destination — route-level analysis"),
    "DEP_TIME":          ("✅ KEEP",    "Raw departure time — source for DEP_HOUR"),
    "DEP_HOUR":          ("✅ KEEP",    "Derived hour — join key + TIME_OF_DAY"),
    "DEP_DELAY":         ("✅ KEEP",    "Raw delay — NEVER overwrite originals"),
    "DEP_DELAY_NEW":     ("✅ KEEP",    "BTS non-negative delay (0 if early)"),
    "DEP_DELAY_CLEAN":   ("✅ KEEP",    "Outlier-capped version for modeling"),
    "CANCELLED":         ("✅ KEEP",    "Binary cancel flag — core outcome variable"),
    "CANCELLATION_CODE": ("✅ KEEP",    "A=Carrier B=Weather C=NAS D=Security"),
    "WEATHER_DELAY":     ("✅ KEEP",    "BTS weather attribution minutes"),
    "CARRIER_DELAY":     ("✅ KEEP",    "BTS carrier attribution minutes"),
    "NAS_DELAY":         ("✅ KEEP",    "National Airspace System delay minutes"),
    "DISTANCE":          ("✅ KEEP",    "Control variable — longer routes have bigger buffers"),
    "AIR_TIME":          ("✅ KEEP",    "Operational efficiency metric"),
    "OUTCOME":           ("✅ KEEP",    "Four-category label for all charts"),
    "SEASON":            ("✅ KEEP",    "Weather confounder control"),
    "IS_WEEKEND":        ("✅ KEEP",    "Traffic pattern control for H3"),
    "TIME_OF_DAY":       ("✅ KEEP",    "Central to H3 cascade hypothesis"),
    "DELAYED_15":        ("✅ KEEP",    "FAA on-time standard binary flag"),
    "DELAYED_60":        ("✅ KEEP",    "Severe delay binary flag"),
    "ON_TIME":           ("✅ KEEP",    "On-time binary flag for KPI charts"),
    "FLIGHT_TYPE":       ("✅ KEEP",    "Domestic vs International"),
    "AIRLINE_TIER":      ("✅ KEEP",    "Legacy/LCC/ULCC/Regional — core for H2"),
    "WEATHER_KEY":       ("🗂 ARCHIVE", "Join key — keep until merge confirmed good"),
    "TEMP_C":            ("✅ KEEP",    "Temperature control variable"),
    "PRECIP_MM":         ("✅ KEEP",    "Precipitation — H1 predictor"),
    "WIND_SPEED_KMH":    ("✅ KEEP",    "Wind speed — H1 predictor"),
    "WIND_GUST_KMH":     ("✅ KEEP",    "Wind gusts — H1 predictor"),
    "VISIBILITY_KM":     ("✅ KEEP",    "Visibility — H1 predictor"),
    "SNOW_MM":           ("✅ KEEP",    "Snowfall — H1 predictor"),
    "PRESSURE_HPA":      ("✅ KEEP",    "Atmospheric pressure control"),
    "SEVERITY_SCORE":    ("✅ KEEP",    "Composite weather index 0–10"),
    "ADVERSE_WEATHER":   ("✅ KEEP",    "Binary adverse conditions flag"),
}

print(f"{'#':<4} {'Column':<25} {'Status':<15} Reason")
print("─" * 85)
for i, col in enumerate(df_merged.columns, 1):
    if col in COL_GUIDE:
        status, reason = COL_GUIDE[col]
        print(f"{i:<4} {col:<25} {status:<15} {reason}")
    else:
        print(f"{i:<4} {col:<25} ❓ UNMAPPED      Not in guide — review manually")

archive = [c for c,v in COL_GUIDE.items() if "ARCHIVE" in v[0] and c in df_merged.columns]
print(f"\n⬇️  Candidates to archive (approve first): {archive}")
print(f"\nTotal columns: {df_merged.shape[1]}")

In [ ]:
# ── Save Output Files ────────────────────────────────────────────────
# THREE FILES:
#
# 1. nyc_flights_weather_full_2021_2026.csv
#    Complete dataset — used for EDA and time-series charts.
#
# 2. nyc_flights_train_2021_2025.csv
#    Training set for predictive model. 2026 is excluded.
#
# 3. nyc_flights_test_2026.csv
#    Holdout set — NEVER shown to the model during training.
#    Used only for final prediction vs actual comparison.

print("Saving files to Google Drive (may take a few minutes)...")

path_full  = os.path.join(CLEAN_DIR, "nyc_flights_weather_full_2021_2026.csv")
path_train = os.path.join(CLEAN_DIR, "nyc_flights_train_2021_2025.csv")
path_test  = os.path.join(CLEAN_DIR, "nyc_flights_test_2026.csv")

# Full dataset
df_merged.to_csv(path_full, index=False)
print(f"✅ Full   : {len(df_merged):,} rows → {path_full.split('/')[-1]}")

# Training set
train_df = df_merged[df_merged["YEAR"] <= 2025].copy()
train_df.to_csv(path_train, index=False)
print(f"✅ Train  : {len(train_df):,} rows → {path_train.split('/')[-1]}")

# Test / holdout set
test_df = df_merged[df_merged["YEAR"] == 2026].copy()
if len(test_df) > 0:
    test_df.to_csv(path_test, index=False)
    print(f"✅ Test   : {len(test_df):,} rows → {path_test.split('/')[-1]}")
else:
    print("⚠️  No 2026 data — test file not created")

print(f"\n💾 All saved to: {CLEAN_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════
# FINAL SUMMARY — Pipeline Complete
# ════════════════════════════════════════════════════════

print("=" * 62)
print("PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 62)

print(f"\n📦 Dataset sizes:")
print(f"   Full  (2021-2026): {len(df_merged):,} flights")
print(f"   Train (2021-2025): {len(train_df):,} flights")
print(f"   Test  (2026)     : {len(test_df):,} flights")

print(f"\n✈️  By Airport:")
for ap in NYC_AIRPORTS:
    sub = df_merged[df_merged["ORIGIN"] == ap]
    act = sub[sub["CANCELLED"] == 0]
    print(f"   {ap}:  {len(sub):>8,} flights | "
          f"cancel={sub['CANCELLED'].mean()*100:.1f}% | "
          f"on-time={act['ON_TIME'].mean()*100:.1f}% | "
          f"mean delay={act['DEP_DELAY'].mean():.1f} min")

print(f"\n🌤  Weather Coverage:")
wx_pct = df_merged["TEMP_C"].notna().mean() * 100
print(f"   Flights with weather data : {wx_pct:.1f}%")
print(f"   Adverse weather rate      : {df_merged['ADVERSE_WEATHER'].mean()*100:.1f}%")
print(f"   Severity score (mean)     : {df_merged['SEVERITY_SCORE'].mean():.2f} / 10")

print(f"\n✅ Ready for analysis:")
print(f"   H1 Predictors  : PRECIP_MM, WIND_SPEED_KMH, SNOW_MM, VISIBILITY_KM, SEVERITY_SCORE")
print(f"   H2 Groups      : AIRLINE_TIER × ADVERSE_WEATHER → DEP_DELAY_CLEAN, CANCELLED")
print(f"   H3 Time        : TIME_OF_DAY → DEP_DELAY_CLEAN (controlling SEASON + ADVERSE_WEATHER)")
print(f"   H4 Airports    : ORIGIN → DEP_DELAY_CLEAN (controlling weather, tier, time)")
print(f"   Predictive     : Train on train_df → predict test_df → compare actual vs predicted")